# 01 — Hamiltonian / QUBO 进展总览

## 研究目标

目标不是证明 Hamilton path 的数学最优性，而是从 OLC 图中得到足够好的 read layout，最终能够还原参考序列。

在当前 benchmark 有完整参考答案与 exact certificate，因此暂时用 ground-state / weighted-path score 作为受控 proxy：

$$
W(P)=\sum_{e\in P}w_e.
$$

评价时应区分三层：

$$
\text{optimization quality}
\neq
\text{layout quality}
\neq
\text{assembly quality}.
$$

因此“没有 ground hit”不等于 assembly 无法使用。

## 1. 静态 QUBO 设计的演化

### 1.1 传统位置变量

传统 Hamilton path / cycle 形式使用

$$
x_{v,j}\in\{0,1\},
$$

表示顶点 $v$ 是否处在位置 $j$。

变量数：

$$
O(N^2).
$$

优点是语义直接；缺点是对稀疏 OLC 图浪费大量变量。

---

### 1.2 Edge-position 原始模型

对每条候选边建立 selection / position bits，CHM13 complex144 上：

$$
N=144,\quad M=261,\quad K=8.
$$

原始模型：

$$
Q=2637.
$$

但 order square 中存在 $2^k$ 系数，导致：

$$
\max |J|\approx 9.44\times10^6.
$$

主要问题不是变量数，而是 **图论上的小 move 在 QUBO Hamming 空间中非常大**。

---

### 1.3 独立 selection + edge-position certificate

将 edge selection $x_e$ 与位置 bits 分离，得到约：

$$
Q=3420.
$$

QbSolv 的 edge topology 明显改善，但位置/order certificate 仍形成高 barrier。

---

### 1.4 bounded vertex-order 系列

曾测试：

- bounded +1：约 7704 variables；
- latent +1：约 5616 variables。

latent +1 虽然系数范围改善，但有限 penalty 被 solver 找到可利用的非法低能态，因此不能视为已验证等价 formulation。

---

### 1.5 strict latent-rank comparator

核心约束改成：

$$
x_{uv}=1\Rightarrow P_v>P_u.
$$

利用 ripple-borrow comparator，不要求精确 $+1$。

CHM13：

$$
Q=3789,
$$

且系数规模大幅缩小：

$$
\max |J|=576.
$$

后来补充 endpoint distinct：

$$
A_{st}\sum_v s_vt_v
$$

以排除 source=sink 导致的“孤立点 + 剩余 cycle”病态解。

静态 black-box solver 仍存在明显的 edge/rank coordination barrier。

## 2. Endpoint-free 3501-variable 路线

去掉显式 source/sink，使用：

1. 每个顶点最多一个 incoming；
2. 每个顶点最多一个 outgoing；
3. selected edge 必须严格沿 rank 增长；
4. 最大化 selected edge 数。

若选中图无环且满足 in/out degree $\le1$，它是若干 directed paths 的并。

若进一步有：

$$
|E_x|=N-1,
$$

则 path cover 只有一个 component，因此得到 Hamilton path。

变量数：

$$
Q=3501.
$$

线性 cardinality reward 版本 QUBO 稀疏；exact count-square 版本

$$
H_{\mathrm{count}}
=
A_{\mathrm{count}}
\left(\sum_e x_e-(N-1)\right)^2
$$

不增加变量，却把 edge bits 近乎全连接，使 quadratic terms 增到约 45872。

静态 QbSolv 可以逼近 143 edges，但会以 degree conflict / order violation 换 cardinality，因此没有解决问题。

## 3. 关键转折：Projected edge-only hybrid

最大的结构变化是：**order/rank 不再交给 annealer 搜索。**

真正 sampler 的只有：

$$
x_e\in\{0,1\},\qquad e\in E,
$$

即 CHM13 只有：

$$
261
$$

个 binary variables。

经典侧维护顶点 rank：

$$
r_v\in\{0,\ldots,N-1\}.
$$

给定当前 rank，定义 backward candidate set：

$$
B^{(t)}
=
\{(u,v)\in E:r_v^{(t)}\le r_u^{(t)}\}.
$$

然后下一轮加入：

$$
H_{\mathrm{order}}^{(t)}
=
A_{\mathrm{ord}}^{(t)}
\sum_{e\in B^{(t)}}x_e.
$$

循环：

$$
r^{(t)}
\to
Q^{(t)}(x)
\xrightarrow{\text{SQA/Tabu}}
x^{(t+1)}
\to
\text{classical rerank}
\to
r^{(t+1)}.
$$

这不是“一次固定 full-QUBO 的纯 SQA”，而是 adaptive-QUBO hybrid。

## 4. Projected hybrid 的实际结果

### OpenJij SQA

edge-only projected SQA 能反复得到合法 Hamilton path。

扩展搜索中最好 weighted score：

$$
1{,}327{,}035.
$$

认证 optimum：

$$
1{,}343{,}093.
$$

gap：

$$
16{,}058\approx1.20\%.
$$

该非最优 path 有一个 read 的最大顺序位移达到 97，但用相同粗 overlap-splice 方法重建序列后，reference coverage / identity 与 reference-order control 极接近。这说明 weighted ground-state gap 与最终 assembly error 并非简单线性关系。

### Projected Tabu

相同 projected framework 中，Tabu 曾得到：

$$
\boxed{1{,}343{,}093}
$$

即 certified optimum。

这支持“主要瓶颈是 solver–representation mismatch”而不是单纯 Hamiltonian 错误。

## 5. 当前 sampler 研究：两个状态空间路线

### Route A：Full space（当前优先）

状态空间保持：

$$
\Omega=\{0,1\}^{261}.
$$

Hamiltonian 保留：

$$
H=
H_{\mathrm{weight}}
+
H_{\mathrm{degree}}
+
H_{\mathrm{count}}
+
H_{\mathrm{order}}.
$$

其中：

$$
H_{\mathrm{count}}
=
A_{\mathrm{count}}
\left(\sum_e x_e-143\right)^2.
$$

sampler 可以混合：

- single-bit；
- exchange；
- reconnect；
- insertion/removal；
- 后续 defect-aware multi-bit move。

只要 proposal 对目标 path-integral distribution 保持 detailed balance，multi-bit MCMC move 本身不改变目标 quantum Hamiltonian。

---

### Route B：Hard fixed-cardinality（算法对照）

直接限制：

$$
\Omega_{143}
=
\left\{x:\sum_e x_e=143\right\}.
$$

则：

$$
H_{\mathrm{count}}\equiv0.
$$

可以删掉 dense count square。

这条路线搜索效率好，但严格受限状态空间不再等同于标准 independent transverse-field driver 的完整状态空间，因此作为 constrained / SQA-inspired branch 保留。

## 6. Exchange 与 reconnect：术语及功能

### Exchange

以后写成 $k_{\mathrm{ex}}$。

一次 move：

$$
k_{\mathrm{ex}}\text{ 个 }1\to0
+
k_{\mathrm{ex}}\text{ 个 }0\to1.
$$

$k_{\mathrm{ex}}=1$ 是 2-bit move；$k_{\mathrm{ex}}=3$ 是 6-bit move。

它能改变各顶点 degree profile。

### Reconnect

当前 reconnect 是独立 kernel，不叫 $k$：

$$
(a\to b,c\to d)
\leftrightarrow
(a\to d,c\to b).
$$

它保持：

$$
|E_x|,
\quad
d_v^{in},
\quad
d_v^{out}.
$$

因此它不适合早期修 degree，但适合在 degree 已接近正确以后换拓扑 / 解 cycle。

## 7. Fixed-cardinality sampler 的关键实验

一个成功 seed 展示了很清楚的 phase behavior：

先用 $k_{\mathrm{ex}}=1$ 修 degree：

$$
\text{random 143-edge state}
\to
\text{Hamilton path},
$$

得到 score：

$$
1{,}326{,}008.
$$

然后开启 reconnect，状态暂时经过含 cycle 但 degree 正确的中间态，最终：

$$
1{,}326{,}008
\to
\boxed{1{,}343{,}093}.
$$

因此当前最合理的角色分工是：

$$
\boxed{\text{exchange：修 degree}}
$$

$$
\boxed{\text{reconnect：换 topology / cycle}}
$$

而不是一开始就大量 reconnect。

## 8. $k_{\mathrm{ex}}>1$ 的结果

已测试：

$$
k_{\mathrm{ex}}=2,3
$$

以及后期多次独立 proposal。

总体接受率大致：

$$
k=1:\sim1\%,
\qquad
k=2:\sim0.4\%,
\qquad
k=3:\sim0.26\%.
$$

即使在后期把随机 $k=2,3$ proposal 数增加到 baseline 的数倍，也没有稳定突破最后的 degree barrier。

当前判断：

$$
\boxed{\text{问题不是“随机大步次数太少”，而是随机大步缺乏结构。}}
$$

后续若继续 $k>1$，应围绕 degree-conflict vertex、cycle boundary、component endpoint 构造 defect-aware proposal。

## 9. Full-space 第一轮结果：新的 cardinality barrier

恢复 full-space 与 $H_{\mathrm{count}}$ 后：

- degree conflict 很容易降到 0；
- 但 selected edge count 常停在 139–142；
- single + exchange / reconnect 不能稳定补回最后 1–4 条边。

因此 full-space 的核心问题变成：

$$
\boxed{\text{cardinality + degree 的联合能垒}}
$$

而不是单纯 degree。

已加入一种 3-bit insertion/removal move：

$$
\{a\to b\}
\leftrightarrow
\{a\to c,c\to b\}.
$$

它允许：

$$
\Delta |E_x|=\pm1
$$

且 proposal 可做成 static involution。

complex144 中存在 134 个此类 template；eligibility 很高，但接受率仍约 $10^{-3}$ 以下，暂未得到 Hamilton path。

下一步应设计更强的：

$$
k\leftrightarrow k+1
$$

局部 repair，使 cardinality 改变与 degree/topology 修复同时发生。

## 10. 当前结论与下一步

当前不再以“继续减少变量数”为首要目标。

优先问题是：

$$
\boxed{\text{如何让一次 sampler move 对应一次有意义的 OLC topology move}}
$$

下一步优先级：

1. full-space defect-aware cardinality-changing moves；
2. 优化 reconnect，而非继续增加随机 $k_{\mathrm{ex}}>1$ 次数；
3. 保留 hard fixed-cardinality 作为性能对照；
4. reverse annealing 暂缓，后续专门研究；
5. compression 暂时不作为 100–150 read 规模图的默认步骤。